In [1]:
import os
import pandahouse
import numpy as np
from scipy import stats
from tqdm.auto import tqdm


# Сценарий 1. Базовая оценка мощности A/B-теста

**Задача:** оценить вероятность обнаружения изменения среднего количества лайков на пользователя, если алгоритм добавляет 1–2 просмотра с вероятностью 90% только пользователям с 50 и более просмотрами.

**Параметры:** недельный эксперимент, случайное распределение 50/50, 20 000 симуляций, t-тест Уэлча при α = 0,05.

Исходный код сохранён с результатами вычислений. Доступ к учебной базе завершён, поэтому повторный запуск требует восстановления исходных распределений. Пароли удалены.


## Подход

По частотным распределениям просмотров и CTR из ClickHouse формируются две виртуальные группы пользователей. Эффект алгоритма задаётся дополнительными просмотрами в тестовой группе, а лайки моделируются биномиальным распределением.


## Данные

Для оценки исходных распределений использовались события ленты за 22–28 мая 2026 года. Просмотры агрегировались по пользователям за неделю, CTR в исходном запросе рассчитывался на уровне пользователя и дня. Это различие необходимо учитывать при интерпретации мощности.


In [2]:
# Параметры подключения считываются из переменных окружения.
# Для запуска потребуются действующие учётные данные ClickHouse.
connection = {
    'host': os.getenv('CH_HOST'),
    'user': os.getenv('CH_USER'),
    'password': os.getenv('CH_PASSWORD'),
    'database': os.getenv('CH_DATABASE', 'simulator_20260620'),
}


In [3]:
# Запрос 1: Распределение просмотров
q_views = """
SELECT views, count() as users
FROM (
    SELECT user_id,
    SUM(action = 'view') as views
    FROM {db}.feed_actions
    WHERE toDate(time) BETWEEN '2026-05-22' AND '2026-05-28'
    GROUP BY user_id
)
GROUP BY views
ORDER BY views
"""

In [4]:
# Запрос 2: Распределение CTR пользователей
q_ctr = """
SELECT floor(ctr, 2) as ctr, count() as users
FROM (
    SELECT user_id,
    toDate(time) AS dt,
    SUM(action = 'like') / SUM(action = 'view') as ctr
    FROM {db}.feed_actions
    WHERE toDate(time) BETWEEN '2026-05-22' AND '2026-05-28'
    GROUP BY dt, user_id
)
GROUP BY ctr
"""

In [5]:
# Сколько пользователей совершили конкретное количество просмотров
views_distribution = pandahouse.read_clickhouse(q_views.format(db=connection['database']), connection=connection)
views_distribution.head()

,views,users
0,1,4
1,2,1
2,3,4
3,4,5
4,5,18


In [6]:
# Сколько пользователей имеют конкретный CTR

ctr_distribution = pandahouse.read_clickhouse(q_ctr.format(db=connection['database']), connection=connection)
ctr_distribution.head()

,ctr,users
0,0.00,1443
1,0.65,4
2,0.71,5
3,0.49,4
4,0.54,72


## Параметры моделирования

Размер контрольной и тестовой групп оценивается как половина пользователей в историческом периоде 22–28 мая 2026 года. Вероятности выборки просмотров и CTR определяются из частотных распределений.


In [7]:
# Рассчитываем размер одной группы (50/50 от общего числа пользователей)
group_size = int(views_distribution['users'].sum() // 2)

In [8]:
group_size

20998

In [9]:
# Считаем относительные вероятности (какова веротяность что будет пользователь с таким количсетвом просмотров/лайков)
prob_views = views_distribution['users'] / views_distribution['users'].sum()
prob_ctr = ctr_distribution['users'] / ctr_distribution['users'].sum()
prob_views

0      0.000095
1      0.000024
2      0.000095
3      0.000119
4      0.000429
         ...   
296    0.000024
297    0.000024
298    0.000024
299    0.000024
300    0.000024
Name: users, Length: 301, dtype: float64

In [10]:
# Переводим данные в быстрые одномерные массивы Numpy
array_views = views_distribution['views'].to_numpy()
array_ctr = ctr_distribution['ctr'].to_numpy()

## Моделирование

На каждой итерации моделируется активность обеих групп. Для тестовой группы добавляются 1–2 просмотра с вероятностью 90% только пользователям с 50 и более просмотрами. Лайки генерируются биномиальным распределением. Затем выполняется t-тест Уэлча.

В сохранённом запуске `seed` не фиксировался.


In [11]:
# Инициализируем генератор случайных чисел и пустой список для p-values
rng = np.random.default_rng()  # Исходный запуск: без фиксированного seed
n_simulations = 20000
p_values = []

# Запускаем симуляцию Монте-Карло с отслеживанием прогресса
for _ in tqdm(range(n_simulations), desc="Расчет мощности"):
    
    # Сэмплируем просмотры для групп А и В
    sample_A_views = rng.choice(array_views, size=group_size, replace=True, p=prob_views)
    sample_B_views = rng.choice(array_views, size=group_size, replace=True, p=prob_views)
    
    # Сэмплируем CTR для групп А и В
    sample_A_ctr = rng.choice(array_ctr, size=group_size, replace=True, p=prob_ctr)
    sample_B_ctr = rng.choice(array_ctr, size=group_size, replace=True, p=prob_ctr)
    
    # Моделируем эффект ML-алгоритма на просмотры в Группе В
    sample_B_views_affected = sample_B_views + (
        rng.binomial(n=1, p=0.9, size=group_size) * 
        (1 + rng.binomial(n=1, p=0.5, size=group_size)) * 
        (sample_B_views >= 50)
    )
    
    # Генерируем число лайков биномиальным распределением
    # .astype(np.int64) необходим, так как n в биномиальном распределении должно быть строго целым числом
    group_A_likes = rng.binomial(n=sample_A_views.astype(np.int64), p=sample_A_ctr)
    group_B_likes = rng.binomial(n=sample_B_views_affected.astype(np.int64), p=sample_B_ctr)
    
    # Сравниваем среднее количество лайков на пользователя с помощью t-теста Уэлча
    _, p_value = stats.ttest_ind(group_A_likes, group_B_likes, equal_var=False)
    p_values.append(p_value)

# Превращаем результат в массив Numpy
p_values = np.array(p_values)

## Результат

Мощность оценивается как доля итераций с `p-value < 0.05`. Это вероятность обнаружить смоделированный эффект в заданных условиях, а не фактический результат проведённого A/B-теста.


In [12]:
power = np.mean(p_values < 0.05)
print(f'Мощность: {power * 100:.1f}%')


Мощность: 25.8%


## Ограничения сохранённого запуска

В коде различается уровень агрегации просмотров (неделя на пользователя) и CTR (день на пользователя). Это может влиять на оценку мощности. Перед использованием подхода для планирования реального эксперимента запросы следует согласовать и выполнить повторную проверку.

Код подключения в публичной версии изменён, исходные численные результаты сохранены без пересчёта.
